# Stream A — QC + Feature Analysis

Three tasks:
1. **Waveform QC** — raw audio comparison, normal vs depressed subjects (defined by PHQ-8), annotate abnormal
2. **F0 + Energy QC plot** — pitch variability alongside energy features across all subjects
3. **Z-score normalization** — standardize all features before classification

---
## Cell 0 — Configuration

In [ ]:
from pathlib import Path

# ✏️ Edit if paths differ
RAW_DATA_DIR  = Path("/Users/Meihui/Downloads/sync/work/whisperx")
FEATURES_CSV  = Path("outputs/features/improved_features.csv")
LABELS_DIR    = RAW_DATA_DIR
QC_OUTPUT_DIR = Path("outputs/qc")
QC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PHQ-8 threshold: >= 10 = depressed
PHQ8_THRESHOLD = 10
TARGET_SR      = 16000
WAVEFORM_SECS  = 60   # how many seconds to load for waveform comparison

print("Configuration loaded")
print(f"  Features CSV : {FEATURES_CSV}")
print(f"  QC output    : {QC_OUTPUT_DIR.resolve()}")

---
## Cell 1 — Load Features + PHQ-8 Labels

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# --- Load feature matrix ---
df = pd.read_csv(FEATURES_CSV)
print(f"Features loaded: {df.shape[0]} subjects x {df.shape[1]} columns")

# --- Load labels if not already merged ---
if "PHQ8_Score" not in df.columns:
    dfs = []
    for fname in ["train_split_Depression_AVEC2017.csv",
                  "dev_split_Depression_AVEC2017.csv"]:
        fpath = LABELS_DIR / fname
        if fpath.exists():
            tmp = pd.read_csv(fpath)
            tmp["subject_id"] = tmp["Participant_ID"].astype(str) + "_P"
            dfs.append(tmp)
    if dfs:
        labels = pd.concat(dfs)[["subject_id", "PHQ8_Binary", "PHQ8_Score", "Gender"]]
        df = df.merge(labels, on="subject_id", how="left")
        print(f"Labels merged: {df['PHQ8_Score'].notna().sum()} subjects with PHQ-8")
    else:
        print("WARNING: Label files not found — PHQ-8 columns will be empty")
        df["PHQ8_Score"]  = np.nan
        df["PHQ8_Binary"] = np.nan
        df["Gender"]      = np.nan
else:
    print("PHQ-8 labels already in CSV")

# --- Classify each subject ---
df["group"] = df["PHQ8_Score"].apply(
    lambda x: "depressed" if pd.notna(x) and x >= PHQ8_THRESHOLD
              else ("normal" if pd.notna(x) else "unknown")
)

print(f"\nSubject breakdown:")
print(df[["subject_id", "PHQ8_Score", "PHQ8_Binary", "Gender", "group"]].to_string(index=False))

---
## Cell 2 — Task 1: Waveform QC
Compare raw audio of depressed vs normal subjects. Annotate outliers.

In [ ]:
import librosa
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

def find_audio(subject_id: str) -> Path:
    d = RAW_DATA_DIR / subject_id
    session_id = subject_id.replace("_P", "")
    audio = d / f"{session_id}_AUDIO.wav"
    if not audio.exists():
        wavs = sorted(d.glob("*.wav"))
        if wavs:
            return wavs[0]
    return audio


def compute_waveform_stats(subject_id: str) -> dict:
    """Load audio and return amplitude statistics."""
    path = find_audio(subject_id)
    if not path.exists():
        return None
    audio, _ = librosa.load(str(path), sr=TARGET_SR, mono=True,
                             duration=WAVEFORM_SECS)
    rms = librosa.feature.rms(y=audio)[0]
    return {
        "audio"    : audio,
        "peak"     : float(np.max(np.abs(audio))),
        "rms_mean" : float(np.mean(rms)),
        "rms_std"  : float(np.std(rms)),
    }


# --- Compute stats for all subjects ---
waveform_data = {}
for _, row in df.iterrows():
    sid   = row["subject_id"]
    stats = compute_waveform_stats(sid)
    if stats:
        stats["group"]      = row["group"]
        stats["phq8"]       = row["PHQ8_Score"]
        waveform_data[sid]  = stats

# --- Identify amplitude outliers (>2 SD from group mean) ---
rms_values = np.array([v["rms_mean"] for v in waveform_data.values()])
rms_mean   = np.mean(rms_values)
rms_std    = np.std(rms_values)
for sid, stats in waveform_data.items():
    stats["amplitude_outlier"] = abs(stats["rms_mean"] - rms_mean) > 2 * rms_std

# --- Pick representative normal subject (lowest PHQ-8 with data) ---
normal_subjects = df[df["group"] == "normal"].sort_values("PHQ8_Score")
depressed_subjects = df[df["group"] == "depressed"].sort_values(
    "PHQ8_Score", ascending=False)

ref_sid  = normal_subjects.iloc[0]["subject_id"] if len(normal_subjects) > 0 else df.iloc[0]["subject_id"]
comp_sid = depressed_subjects.iloc[0]["subject_id"] if len(depressed_subjects) > 0 else df.iloc[1]["subject_id"]

ref_phq8  = df[df["subject_id"] == ref_sid]["PHQ8_Score"].values[0]
comp_phq8 = df[df["subject_id"] == comp_sid]["PHQ8_Score"].values[0]

print(f"Reference (normal)   : {ref_sid}  PHQ-8={ref_phq8}")
print(f"Comparison (depressed): {comp_sid}  PHQ-8={comp_phq8}")

# --- Plot: waveform comparison ---
fig, axes = plt.subplots(3, 2, figsize=(16, 11))
pairs     = [(ref_sid, "normal", "steelblue"), (comp_sid, "depressed", "tomato")]

for col, (sid, group, color) in enumerate(pairs):
    stats = waveform_data.get(sid)
    if not stats:
        continue
    audio  = stats["audio"]
    t      = np.linspace(0, len(audio) / TARGET_SR, len(audio))
    phq8   = stats["phq8"]
    outlier_note = " ⚠️ AMPLITUDE OUTLIER" if stats["amplitude_outlier"] else ""

    # Row 0: raw waveform
    axes[0, col].plot(t, audio, color=color, linewidth=0.3, alpha=0.8)
    axes[0, col].set_ylim(-1.05, 1.05)
    axes[0, col].set_title(
        f"{sid}  |  PHQ-8 = {phq8}  |  {group.upper()}{outlier_note}",
        fontsize=11, fontweight="bold", color=color
    )
    axes[0, col].set_ylabel("Amplitude")
    axes[0, col].set_xlabel("Time (s)")
    axes[0, col].text(
        0.02, 0.92,
        f"Peak={stats['peak']:.3f}  RMS={stats['rms_mean']:.4f}",
        transform=axes[0, col].transAxes, fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
    )

    # Row 1: RMS energy over time
    rms_frames = librosa.feature.rms(y=audio, frame_length=1024, hop_length=512)[0]
    rms_times  = librosa.frames_to_time(
        np.arange(len(rms_frames)), sr=TARGET_SR, hop_length=512)
    axes[1, col].plot(rms_times, rms_frames, color=color, linewidth=0.8)
    axes[1, col].axhline(np.mean(rms_frames), color="black",
                         linestyle="--", linewidth=1,
                         label=f"mean={np.mean(rms_frames):.4f}")
    axes[1, col].set_title(f"{sid} — RMS Energy over Time", fontsize=10)
    axes[1, col].set_ylabel("RMS")
    axes[1, col].set_xlabel("Time (s)")
    axes[1, col].legend(fontsize=8)

    # Row 2: amplitude histogram
    axes[2, col].hist(audio, bins=100, color=color, alpha=0.7, edgecolor="none")
    axes[2, col].set_title(f"{sid} — Amplitude Distribution", fontsize=10)
    axes[2, col].set_xlabel("Amplitude")
    axes[2, col].set_ylabel("Count")

plt.suptitle(
    f"Waveform QC: Normal (PHQ-8={ref_phq8}) vs Depressed (PHQ-8={comp_phq8})\n"
    "Amplitude outliers (>2 SD from group mean) flagged with ⚠️",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
out = QC_OUTPUT_DIR / "qc_waveform_phq8.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {out}")

# --- Print amplitude outlier summary ---
print("\nAmplitude outlier summary:")
for sid, stats in waveform_data.items():
    flag = " <-- OUTLIER ⚠️" if stats["amplitude_outlier"] else ""
    print(f"  {sid:<12}  PHQ-8={stats['phq8']:>5}  "
          f"RMS={stats['rms_mean']:.4f}  "
          f"group={stats['group']:<10}{flag}")

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

def diagnose_recording(subject_id: str):
    # Load first 120 seconds of audio at 16kHz
    path = find_audio(subject_id)
    audio, _ = librosa.load(str(path), sr=16000, mono=True, duration=120)
    
    # Compute amplitude statistics
    peak         = np.max(np.abs(audio))
    rms          = np.sqrt(np.mean(audio**2))
    clipping_pct = np.mean(np.abs(audio) > 0.99) * 100  # % of samples near max
    
    print(f"{'='*45}")
    print(f"Subject       : {subject_id}")
    print(f"Peak amplitude: {peak:.4f}  (1.0 = max possible)")
    print(f"RMS           : {rms:.4f}")
    print(f"Clipping %    : {clipping_pct:.3f}%  (>0.1% = suspicious)")
    print(f"{'='*45}")
    
    # Verdict: clipping means waveform was truncated (unrecoverable),
    # high amplitude means gain issue but waveform is intact (fixable with normalize)
    if clipping_pct > 0.1:
        print("VERDICT: ⚠️  Likely clipping — mic too close or gain too high")
        print("         Recommend: peak normalize before feature extraction")
    elif peak > 0.95:
        print("VERDICT: ⚠️  Very high amplitude — possible gain issue")
    else:
        print("VERDICT: ✅  Amplitude looks normal")
    
    # Plot waveform for visual inspection
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    t = np.linspace(0, len(audio)/16000, len(audio))
    
    # Top: full waveform — check overall amplitude level
    axes[0].plot(t, audio, linewidth=0.3, color="steelblue")
    axes[0].axhline( 0.99, color="red", linestyle="--", linewidth=1, label="clipping threshold")
    axes[0].axhline(-0.99, color="red", linestyle="--", linewidth=1)
    axes[0].set_ylim(-1.05, 1.05)
    axes[0].set_title(f"{subject_id} — Full Waveform (peak={peak:.3f}, clipping={clipping_pct:.3f}%)")
    axes[0].set_xlabel("Time (s)")
    axes[0].legend(fontsize=8)
    
    # Bottom: first 5 seconds zoomed in — look for flat-topped waveform indicating clipping
    n5 = 5 * 16000
    axes[1].plot(t[:n5], audio[:n5], linewidth=0.5, color="tomato")
    axes[1].axhline( 0.99, color="red", linestyle="--", linewidth=1)
    axes[1].axhline(-0.99, color="red", linestyle="--", linewidth=1)
    axes[1].set_ylim(-1.05, 1.05)
    axes[1].set_title(f"{subject_id} — First 5 seconds (zoom in to check clipping)")
    axes[1].set_xlabel("Time (s)")
    
    plt.tight_layout()
    plt.savefig(QC_OUTPUT_DIR / f"waveform_{subject_id}.png", dpi=150)
    plt.show()


# --- Scan all subjects before plotting, to get a full summary table first ---
results = []
for sid in df["subject_id"].tolist():
    path = find_audio(sid)
    audio, _ = librosa.load(str(path), sr=16000, mono=True, duration=120)
    peak         = float(np.max(np.abs(audio)))
    rms          = float(np.sqrt(np.mean(audio**2)))
    clipping_pct = float(np.mean(np.abs(audio) > 0.99) * 100)
    results.append({
        "subject_id"  : sid,
        "PHQ8_Score"  : df[df["subject_id"]==sid]["PHQ8_Score"].values[0],
        "peak"        : peak,
        "rms"         : rms,
        "clipping_pct": clipping_pct,
        # clipping = waveform truncated (unrecoverable)
        # high amp = gain issue only (fixable)
        "verdict"     : "⚠️ CLIPPING" if clipping_pct > 0.1
                        else ("⚠️ HIGH AMP" if peak > 0.95 else "✅ OK"),
    })

result_df = pd.DataFrame(results).sort_values("clipping_pct", ascending=False)
print(result_df.to_string(index=False))

# Only plot waveforms for flagged subjects to avoid noise
flagged = result_df[result_df["verdict"] != "✅ OK"]
print(f"\nFlagged: {len(flagged)} subjects")
for sid in flagged["subject_id"]:
    diagnose_recording(sid)


---
## Cell 3 — Task 2: F0 + Energy QC Plot
Pitch variability alongside energy. Subjects coloured by PHQ-8 group.

In [ ]:
subjects = df["subject_id"].tolist()
groups   = df["group"].tolist()
phq8s    = df["PHQ8_Score"].tolist()
n        = len(subjects)
x        = np.arange(n)

# Color per subject: blue=normal, red=depressed, grey=unknown
GROUP_COLORS = {"normal": "steelblue", "depressed": "tomato", "unknown": "grey"}
bar_colors   = [GROUP_COLORS[g] for g in groups]

# --- Feature groups to plot ---
FEATURES = [
    # (column_name_in_csv, display_label, group)
    # F0 — pitch
    ("F0semitoneFrom27.5Hz_sma3nz_amean_mean",
     "F0 Mean (semitone)  [F0semitoneFrom27.5Hz_sma3nz_amean]",      "F0"),
    ("F0semitoneFrom27.5Hz_sma3nz_stddevNorm_mean",
     "F0 Std Norm  ← PITCH VARIABILITY  [F0semitoneFrom27.5Hz_sma3nz_stddevNorm]", "F0"),
    ("F0semitoneFrom27.5Hz_sma3nz_percentile20.0_mean",
     "F0 20th Percentile (semitone)",                                 "F0"),
    ("F0semitoneFrom27.5Hz_sma3nz_percentile80.0_mean",
     "F0 80th Percentile (semitone)",                                 "F0"),
    # Energy — loudness
    ("loudness_sma3_amean_mean",
     "Loudness Mean",                                                  "Energy"),
    ("loudness_sma3_stddevNorm_mean",
     "Loudness Std Norm",                                              "Energy"),
    # Voice quality
    ("HNRdBACF_sma3nz_amean_mean",
     "HNR — Harmonics-to-Noise Ratio (voice quality)",                "Quality"),
    ("jitterLocal_sma3nz_amean_mean",
     "Jitter — voice irregularity",                                   "Quality"),
]

available = [(col, label, grp) for col, label, grp in FEATURES
             if col in df.columns]
missing   = [(col, label, grp) for col, label, grp in FEATURES
             if col not in df.columns]
if missing:
    print("Columns not found in CSV (skipped):")
    for col, _, _ in missing:
        print(f"  {col}")

n_plots = len(available)
fig, axes = plt.subplots(n_plots, 1, figsize=(15, n_plots * 3.5))
plt.subplots_adjust(hspace=0.6)
if n_plots == 1:
    axes = [axes]

GROUP_SECTION_COLORS = {"F0": "#FF8C00", "Energy": "#2196F3", "Quality": "#9C27B0"}

for i, (col, label, grp) in enumerate(available):
    vals       = df[col].values.astype(float)
    gmean      = np.nanmean(vals)
    gstd       = np.nanstd(vals)
    is_outlier = np.abs(vals - gmean) > 2 * gstd

    # Use group colors; override with orange if amplitude outlier
    colors_i = ["orange" if is_outlier[j] else bar_colors[j]
                for j in range(n)]

    bars = axes[i].bar(x, vals, color=colors_i, alpha=0.85,
                       edgecolor="white", linewidth=0.5)

    # Annotate outlier bars
    for j, (val, outlier) in enumerate(zip(vals, is_outlier)):
        if outlier:
            offset = 0.03 * (np.nanmax(vals) - np.nanmin(vals))
            axes[i].text(j, val + offset, "⚠️", ha="center",
                         va="bottom", fontsize=9)

    # Annotate PHQ-8 score below each bar
    for j, (sid, phq) in enumerate(zip(subjects, phq8s)):
        label_txt = f"{phq:.0f}" if pd.notna(phq) else "?"
        axes[i].annotate(
            label_txt,
            xy=(j, 0), xycoords=("data", "axes fraction"),
            xytext=(0, -22), textcoords="offset points",
            ha="center", va="top", fontsize=6.5, color="dimgrey"
        )
    axes[i].axhline(gmean, color="black", linestyle="--",
                    linewidth=1.0, label=f"mean={gmean:.4f}")
    axes[i].axhspan(gmean - gstd, gmean + gstd,
                    alpha=0.07, color="black", label="±1 SD")
    section_color = GROUP_SECTION_COLORS.get(grp, "black")
    axes[i].set_title(f"[{grp}]  {label}",
                      fontsize=9, fontweight="bold",
                      color=section_color, loc="left")
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(subjects, rotation=35, ha="right", fontsize=8)
    axes[i].legend(fontsize=7, loc="upper right")

# Legend for group colors
legend_patches = [
    mpatches.Patch(color="steelblue", label="Normal (PHQ-8 < 10)"),
    mpatches.Patch(color="tomato",    label="Depressed (PHQ-8 ≥ 10)"),
    mpatches.Patch(color="orange",    label="Outlier (>2 SD)"),
    mpatches.Patch(color="grey",      label="Unknown"),
]
fig.legend(handles=legend_patches, loc="upper right",
           bbox_to_anchor=(1.15, 1.0), fontsize=9, framealpha=0.9)

plt.suptitle(
    "QC: F0 (Pitch) + Energy Features\n"
    "Bar color = PHQ-8 group  |  ⚠️ = >2 SD outlier  |  Number below bar = PHQ-8 score",
    fontsize=12, fontweight="bold",
    y=1.02        
)

plt.tight_layout()
out = QC_OUTPUT_DIR / "qc_f0_energy_phq8.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {out}")

---
## Cell 4 — Task 3: Z-score Normalization
Standardize all features before classification. RMS and MFCC are on completely different scales.

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- Separate feature columns from metadata ---
META_COLS    = ["subject_id", "PHQ8_Binary", "PHQ8_Score",
                "Gender", "split", "group",
                "n_segments", "total_speech_sec"]
meta_cols_present = [c for c in META_COLS if c in df.columns]
feature_cols      = [c for c in df.columns if c not in meta_cols_present]

print(f"Feature columns to normalize : {len(feature_cols)}")
print(f"Meta columns kept as-is      : {meta_cols_present}")

# --- Check scale differences BEFORE normalization ---
print("\nScale check BEFORE z-score (sample features):")
sample_cols = [
    c for c in feature_cols
    if any(k in c for k in ["loudness", "F0semitone", "mfcc", "jitter", "HNR"])
][:8]
for col in sample_cols:
    vals = df[col].dropna()
    print(f"  {col:<55} min={vals.min():>10.4f}  max={vals.max():>10.4f}")

# --- Z-score normalization ---
scaler     = StandardScaler()
X          = df[feature_cols].values.astype(float)
X_scaled   = scaler.fit_transform(X)

df_scaled              = df[meta_cols_present].copy()
df_scaled[feature_cols] = X_scaled

# --- Check scale AFTER normalization ---
print("\nScale check AFTER z-score:")
for col in sample_cols:
    vals = df_scaled[col].dropna()
    print(f"  {col:<55} min={vals.min():>10.4f}  max={vals.max():>10.4f}")

print(f"\nVerification — all feature means ≈ 0:")
means = df_scaled[feature_cols].mean()
print(f"  Max absolute mean: {means.abs().max():.6f}  (should be ~0)")
print(f"  Max absolute std : {df_scaled[feature_cols].std().max():.4f}  (should be ~1)")

In [ ]:
# Verify peak normalization effectiveness: check if HIGH AMP subjects (305_P, 306_P, 307_P)
# still dominate loudness after normalization. Acceptable if gap < 2x group mean.

loudness_col = "loudness_sma3_amean_mean"
if loudness_col in df.columns:
    print(df[["subject_id", loudness_col]].sort_values(
        loudness_col, ascending=False).to_string(index=False))

---
## Cell 5 — Save Normalized CSV + Final Summary

In [ ]:
# --- Save ---
out_csv = Path("outputs/features/improved_features_zscored.csv")
df_scaled.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")
print(f"  Shape     : {df_scaled.shape}")
print(f"  File size : {out_csv.stat().st_size / 1e3:.1f} KB")

# --- Before/after scale comparison plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample_plot_cols = sample_cols[:6]
before_data = [df[c].dropna().values for c in sample_plot_cols]
after_data  = [df_scaled[c].dropna().values for c in sample_plot_cols]
short_names = [
    c.replace("F0semitoneFrom27.5Hz_sma3nz_", "F0_")
     .replace("loudness_sma3_", "loud_")
     .replace("_mean", "")
    for c in sample_plot_cols
]
axes[0].boxplot(before_data, labels=short_names, patch_artist=True)
axes[0].set_title("Before Z-score", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Raw value")
axes[0].tick_params(axis="x", rotation=30)

axes[1].boxplot(after_data, labels=short_names, patch_artist=True)
axes[1].set_title("After Z-score", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Standardized value")
axes[1].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Feature Scale: Before vs After Z-score Normalization",
             fontsize=12, fontweight="bold")
plt.tight_layout()
scale_plot = QC_OUTPUT_DIR / "qc_zscore_comparison.png"
plt.savefig(scale_plot, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {scale_plot}")

# --- Final summary ---
print("\n" + "=" * 55)
print("QC + Normalization Summary")
print("=" * 55)
print(f"  Subjects          : {len(df_scaled)}")
print(f"  Feature columns   : {len(feature_cols)}")
print(f"  Normal (PHQ-8<10) : {(df['group']=='normal').sum()}")
print(f"  Depressed (≥10)   : {(df['group']=='depressed').sum()}")
print(f"  Unknown           : {(df['group']=='unknown').sum()}")
print()
print("Output files:")
for f in [out_csv] + sorted(QC_OUTPUT_DIR.glob("*.png")):
    print(f"  {f.name}")
print()
print("Ready for classification.")
print("Use: improved_features_zscored.csv")